In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim.lr_scheduler as lr_scheduler
from comet_ml import Experiment
from scipy import linalg
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from src.utils.samplers.data import DatasetSampler, PairedLoaderSampler
from src.utils.samplers.synthetic import StandardNormalSampler, SwissRollSampler
from src.utils.training.weather_notebook import (
    load_weather_tensors,
    paired_sampler_weather,
    unpaired_sampler_weather,
)

from src.utils.notebook_setup import ensure_repo_imports

REPO_ROOT = ensure_repo_imports()

tsne = TSNE(n_components=2, random_state=50)


# Data preparation

### PS
1) Source  
$X \in \mathbb{R}^{N \times d_{1}}, N - \text{number of locations}, d_{1} - \text{features dim}$ \
$x = (\mu, \sigma) - \text{for a given location in June}$ \
$N = 1396, d_{1} = 188$ 

2) $Y \in \mathbb{R}^{N \times M \times d_{2}}, N - \text{number of locations}, M - \text{measurements for a given location in January by day}$ \
$M = [1, 31], d_{2} = 94$ 

In [2]:
##########################################
#-------------- RAW DATA -----------------
##########################################

import numpy as np
import pandas as pd

root = '../tabred/kal/weather'

data = np.load(f'{root}/X_num.npy')
data = np.stack([d for d in data if sum(np.isnan(d)) == 0])
data_csv = pd.read_csv(f'{root}/csv/X_num.csv')
#train_data = data[train_idx]
#test_data = data[test_idx]

target = np.load(f'{root}/Y.npy')
meta = np.load(f'{root}/X_meta.npy')
meta = np.stack([meta[i] for i, d in enumerate(data) if sum(np.isnan(d)) == 0])
meta_csv = pd.read_csv(f'{root}/csv/X_meta.csv')

names = list(data_csv.columns)
names.append('location')
data_new = np.concatenate((data, meta[:, -2].reshape(-1, 1)), axis=1)

In [3]:
#########################################################
#--------------- Month/location splitted ---------------- 
#########################################################
scaler = StandardScaler()

dict_location_src = {}
for d in data_new:
    if d[-2] == 1.0:
        d_new = d[:-7]
        try:
            dict_location_src[d[-1]].append(d_new)
        except KeyError:
            dict_location_src[d[-1]] = []
            dict_location_src[d[-1]].append(d_new)
     

dict_location_src_new = {}
for key in dict_location_src.keys():
    item = dict_location_src[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_src_new[key] = item
dict_location_src = dict_location_src_new
# ------------------------------------------------------------

dict_location_trg = {}
for d in data_new:
    if d[-2] == 6.0:
        d_new = d[:-7]
        try:
            dict_location_trg[d[-1]].append(d_new)
        except KeyError:
            dict_location_trg[d[-1]] = []
            dict_location_trg[d[-1]].append(d_new)
    
dict_location_trg_new = {}
for key in dict_location_trg.keys():
    item = dict_location_trg[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_trg_new[key] = item
dict_location_trg = dict_location_trg_new

print(len(dict_location_trg), len(dict_location_src))

1653 1578


In [4]:
#########################################################
#--------------------- X, Y paired ----------------------
#########################################################

chosen_locs = list(dict_location_trg.keys())[:200]
X_pair_orig, Y_pair_orig = [], []
for key in dict_location_src.keys():
    if key not in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_pair_orig.append(x)
    item_trg = dict_location_trg[key]
    Y_pair_orig.append(item_trg) # sample
X_pair_orig = np.stack(X_pair_orig)


#########################################################
#----------------------- X, Y ---------------------------
#########################################################

# N x 1 x 2D - src
# N x M x D - trg
 
# sampling: 
# b x 1 x 2D,
# b x M x D -> sample -> b x 1 x D

X_orig = []
for key in dict_location_src.keys():
    if key in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_orig.append(x)
X_orig = np.stack(X_orig)

Y_orig = []
for key in dict_location_trg.keys():
    if key in chosen_locs:
        continue
    item_trg = dict_location_trg[key]
    Y_orig.append(item_trg) # sample

In [5]:
print(X_orig.shape, len(Y_orig), X_pair_orig.shape, len(Y_pair_orig))

(1386, 188) 1453 (192, 188) 192


# Running

In [6]:
source_data = X_orig
target_data = Y_orig[0]
X_DIM = source_data.shape[1]
Y_DIM = target_data.shape[1]
#X_DIM = data_set["features"].shape[1]
#Y_DIM = data_set["features"].shape[1]
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 50

N_POTENTIALS = 10
M_POTENTIALS = 1 #10
EPSILON = 1
A_DIAGONAL_INIT = 0.5
L_PAIRED_SAMPLES = len(X_pair_orig)
M_X_UNPAIRED_SAMPLES = 0
N_Y_UNPAIRED_SAMPLES = 0

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

NUM_LABELED = 10
TRAIN_SUBSET_SIZE = 2

PLOT_EVERY = 1000
MAX_STEPS = 20000
CONTINUE = -1

In [7]:
EXP_COST = "MLP_deep_deep"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"EBiEOT-GMM_Batch_Effect_"
    + f"EPSILON_{EPSILON}_"
    + f"N_{N_POTENTIALS}_"
    + f"M_{M_POTENTIALS}_"
    + f"with_{EXP_COST}_"
    + f"cost_included_{EXP_COST_INCLUDED}_"
    + f"N_PAIRED_{NUM_LABELED}_"
    + f"M_UNPAIRED_{len(source_data)}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=NUM_LABELED,
    M_UNPAIRED_SAMPLES=len(source_data),
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)


In [8]:
#pytorch_total_params = sum(p.numel() for p in D.parameters())
#pytorch_total_params

## Ablation Study

In [9]:
def paired_sampler(X_pair, Y_pair, b_size):
    idxs = np.random.randint(low=0, high=len(X_pair)-1, size=b_size)
    x_pair_batch = torch.tensor(X_pair[idxs]).to('cuda')
    y_pair_batch = np.stack([Y_pair[idx][random.randint(0, len(Y_pair[idx])-1)] for idx in idxs])
    y_pair_batch = torch.tensor(y_pair_batch).to('cuda')
    return x_pair_batch.to(torch.float32), y_pair_batch.to(torch.float32)

def unpaired_sampler(X, Y, b_size):
    # UNPAIRED SAMPLER
    idxs = np.random.randint(low=0, high=len(X)-1, size=b_size)
    idxs_y = np.array([len(X) - idx - 1 for idx in idxs])

    x_batch = torch.tensor(X[idxs]).to('cuda')
    y_batch = np.stack([Y[idx][random.randint(0, len(Y[idx])-1)] for idx in idxs_y])
    y_batch = torch.tensor(y_batch).to('cuda')
    return x_batch.to(torch.float32), y_batch.to(torch.float32)

In [10]:
from src.utils.samplers.discrete_ot import OTPlanSampler
from src.utils.datasets.paired import generate_paired_data, get_GT_points, get_paired_sampler
import torch.nn.functional as F
from src.networks.nf.real_nvp import ConditionalRealNVP
from scipy import linalg


In [11]:
device = 'cuda'
T = ConditionalRealNVP(
        features=Y_DIM,
        context_features=X_DIM,
        hidden_context_features=512,
        hidden_features=[128],
        num_blocks_per_layer=4,
        num_layers=5,
        use_volume_preserving=False,
    ).to(device)

T_opt_paired = torch.optim.Adam(T.parameters(), lr=3e-4, weight_decay=0.01)


In [12]:
history = {
        "D_loss": [],
        "G_loss": [],
    }

In [13]:
MAX_STEPS = 10000
experiment = Experiment(project_name="inverse_ot")
experiment.set_name(EXP_NAME)
stats = []
D_loss = []
fids = []
fids2 = []
device = 'cuda'

# Splitting
L_PAIRED_SAMPLES = 90
L_UNPAIRED_SAMPLES = 500
X, Y = X_orig[:L_UNPAIRED_SAMPLES], Y_orig[-L_UNPAIRED_SAMPLES:]
X_pair, Y_pair = X_pair_orig[:L_PAIRED_SAMPLES], Y_pair_orig[:L_PAIRED_SAMPLES]
X_pair_test, Y_pair_test = X_pair_orig[-100:], Y_pair_orig[-100:]


for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
        T_opt_paired.zero_grad()
        X_paired, Y_paired = paired_sampler(X_pair, Y_pair, BATCH_SIZE)

        log_prob = T.log_prob(inputs=Y_paired, context=X_paired)
        T_loss = -log_prob.mean()
        T_loss.backward()
        T_opt_paired.step()
        print(f"Loss: {T_loss.item()}")
        
        """
        with torch.no_grad():
            if step % 1000 == 0:
                for x, y in zip(X_pair_test, Y_pair_test):
                    x = torch.tensor(x).unsqueeze(0).to(device)
                    y = torch.tensor(y).to(device)
                    samples = []
                    sample = T.sample(len(y), x.detach().float()).squeeze()
                    fid_samples = np.array(sample.cpu()) #np.array(torch.cat(samples, dim=0).cpu())
                    fid_samples_2 = np.array(y.cpu())

                    mu1 = np.mean(fid_samples, axis=0)
                    sigma1 = np.cov(fid_samples, rowvar=False)
                    mu2 = np.mean(fid_samples_2, axis=0)
                    sigma2 = np.cov(fid_samples_2, rowvar=False)

                    diff = mu1 - mu2
                    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
                    tr_covmean = np.trace(covmean)
                    fid = (diff.dot(diff) + np.trace(sigma1) +  np.trace(sigma2) - 2 * tr_covmean)
                    fids.append(fid.real)
                    fids2.append(fid.real / np.var(fid_samples_2))
                    
                print(np.mean(fids))
                print(np.mean(fids2))
        """

  0%|                                         | 5/10000 [00:00<12:52, 12.95it/s]

Loss: 124.69519805908203
Loss: 122.06782531738281
Loss: 119.5482406616211
Loss: 117.70692443847656
Loss: 116.25128173828125
Loss: 114.65636444091797
Loss: 113.2208251953125


  0%|                                        | 13/10000 [00:00<07:11, 23.15it/s]

Loss: 111.58411407470703
Loss: 109.9658432006836
Loss: 107.77027893066406
Loss: 104.80323791503906
Loss: 101.0558090209961
Loss: 95.69197082519531
Loss: 88.51145935058594


  0%|                                        | 21/10000 [00:00<06:03, 27.42it/s]

...


 20%|███████▋                              | 2026/10000 [00:55<03:16, 40.51it/s]

Loss: -85.5025634765625
Loss: -87.8707275390625
Loss: -91.20401000976562
Loss: -87.38163757324219
Loss: -88.49962615966797
Loss: -90.78417205810547
Loss: -90.29499053955078
Loss: -89.11942291259766
Loss: -91.82395935058594


 20%|███████▋                              | 2032/10000 [00:55<03:38, 36.49it/s]


Loss: -89.83417510986328
Loss: -90.13851928710938


KeyboardInterrupt: 

In [ ]:
# 1.27, 1.33, 1.28

In [25]:
np.mean([1.27, 1.33, 1.28]), np.std([1.27, 1.33, 1.28])

(1.2933333333333332, 0.02624669291337273)